# p63 — DINOv2 ViT-g residual-stream cache for manim

Streamlined caching notebook built from `vit_registers_replication_2.ipynb`. Produces:

1. Both models (`dinov2_vitg14`, `dinov2_vitg14_reg`) and the same image preprocessing as before.
2. **Activation stream** for `stephen/0111.png` → `streams/stream_{plain,reg}.npy`, shape `(41, N, 1536)`, float16.
   Token layout along N: `[CLS, reg_0..reg_3 (reg model only), patch_0..patch_1368]`.
   `stream[i]` = input to block `i`; `stream[40]` = output of the last block (pre final LayerNorm).
3. **Max-activation grids** for `stephen/0111.png` → `max_grids/{plain,reg}/layer_00.png … layer_40.png`.
   Each is `stream[i, n_skip:].max(over channels)` reshaped to 37×37, min–max scaled *per image*, viridis, native 37×37 px.
   Raw grids also saved as `max_grids/max_grids_{plain,reg}.npy` `(41, 37, 37)` float32.
4. **Streaming version** over every png in `stephen/` and `cat/` →
   `{stephen,cat}/{plain,reg}/layer_nn/frame_mmmm.png` plus `{stephen,cat}/input/frame_mmmm.png` (center-cropped 518px input).
   Frames are numbered sequentially from 0 in sorted-filename order; a `frames.json` in each folder maps frame index → source file.

Everything lands in `p63_cache/`.

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
from PIL import Image
from pathlib import Path
from torchvision import transforms

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda it, **kw: it

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device: {device}')

## 0. Config

In [ ]:
INPUT_ROOT = Path('/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/p63_vit_input_images')
OUT_ROOT   = Path('/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/p63_cache')

KEY_IMAGE      = INPUT_ROOT / 'stephen' / '0111.png'   # single-image cache (steps 2 & 3)
STREAM_FOLDERS = ['stephen', 'cat']                      # streaming export (step 4), processed in this order
STREAM_DTYPE   = np.float16                              # dtype for the saved activation stream .npy

PATCH    = 14
IMG_SIZE = 518
G        = IMG_SIZE // PATCH   # 37

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('input :', INPUT_ROOT)
print('output:', OUT_ROOT)

## 1. Models + image loading (same as the replication notebook)

In [ ]:
model_plain = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14').to(device).eval()
model_reg   = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14_reg').to(device).eval()

MODELS = {'plain': model_plain, 'reg': model_reg}
N_LAYERS = len(model_plain.blocks)          # 40 → 41 stream locations
N_SKIP   = {k: 1 + m.num_register_tokens for k, m in MODELS.items()}   # CLS + registers
print(f'layers: {N_LAYERS}, n_skip: {N_SKIP}')

In [ ]:
normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

def load_image(path, size=IMG_SIZE):
    """Center-crop to square, resize to `size`, ImageNet-normalize. Returns (PIL img, tensor[1,3,H,W])."""
    img = Image.open(path).convert('RGB')
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)).resize((size, size), Image.LANCZOS)
    x = normalize(transforms.ToTensor()(img)).unsqueeze(0)
    return img, x

img, x = load_image(KEY_IMAGE)
x = x.to(device)
plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis('off'); plt.title(KEY_IMAGE.name); plt.show()

## 2. Residual stream capture + max grids

`get_residual_stream` hooks the input of every block plus the output of the last one → `(41, N, D)`.
`max_grids` takes the channel-wise max of the patch tokens at every stream location → `(41, 37, 37)`.

In [ ]:
@torch.no_grad()
def get_residual_stream(model, x):
    """Residual stream at every block boundary, batch dim stripped: (n_blocks+1, N, D)."""
    pre, post, hooks = [], [], []
    for blk in model.blocks:
        hooks.append(blk.register_forward_pre_hook(lambda m, inp: pre.append(inp[0].detach())))
    hooks.append(model.blocks[-1].register_forward_hook(lambda m, inp, out: post.append(out.detach())))
    try:
        model.forward_features(x)
    finally:
        for h in hooks:
            h.remove()
    return torch.stack(pre + post)[:, 0]


def max_grids(stream, n_skip):
    """Channel-wise max of patch tokens at each stream location → (n_blocks+1, G, G) numpy float32."""
    return stream[:, n_skip:].max(dim=-1).values.reshape(-1, G, G).float().cpu().numpy()


def save_viridis_png(grid, path):
    """Min-max scale one 2D grid independently, map through viridis, save at native resolution."""
    g = np.asarray(grid, dtype=np.float32)
    lo, hi = g.min(), g.max()
    g = (g - lo) / (hi - lo) if hi > lo else np.zeros_like(g)
    rgb = (cm.viridis(g)[..., :3] * 255).round().astype(np.uint8)
    Image.fromarray(rgb).save(path)

## 3. Cache `stephen/0111.png`: full stream (.npy) + 41 max-grid pngs per model

In [ ]:
stream_dir = OUT_ROOT / 'streams'
grid_dir   = OUT_ROOT / 'max_grids'
stream_dir.mkdir(exist_ok=True)

grids = {}
for name, model in MODELS.items():
    stream = get_residual_stream(model, x)                                  # (41, N, 1536) on device
    print(f'{name}: stream {tuple(stream.shape)}  max|val| = {stream.abs().max().item():.1f}')

    # --- full activation stream ---
    arr = stream.cpu().numpy().astype(STREAM_DTYPE)
    np.save(stream_dir / f'stream_{name}.npy', arr)
    print(f'   saved stream_{name}.npy  {arr.nbytes / 1e6:.0f} MB  dtype={arr.dtype}')

    # --- max grids: npy + 41 independently scaled viridis pngs ---
    grids[name] = max_grids(stream, N_SKIP[name])                            # (41, 37, 37)
    (grid_dir / name).mkdir(parents=True, exist_ok=True)
    np.save(grid_dir / f'max_grids_{name}.npy', grids[name])
    for i in range(grids[name].shape[0]):
        save_viridis_png(grids[name][i], grid_dir / name / f'layer_{i:02d}.png')
    print(f'   saved {grids[name].shape[0]} pngs → max_grids/{name}/')
    del stream

meta = {
    'image': str(KEY_IMAGE), 'img_size': IMG_SIZE, 'patch': PATCH, 'grid': G,
    'n_layers': N_LAYERS, 'stream_dtype': np.dtype(STREAM_DTYPE).name,
    'n_skip': N_SKIP,
    'token_layout': '[CLS, reg_0..reg_{R-1}, patch_0..patch_{G*G-1}]; stream[i] = input to block i, stream[-1] = last block output (pre final LN)',
}
json.dump(meta, open(OUT_ROOT / 'meta.json', 'w'), indent=2)

In [ ]:
# quick look: max grids along the stream for both models (each panel independently scaled, like the saved pngs)
for name in MODELS:
    fig, axes = plt.subplots(4, 11, figsize=(22, 8))
    for i, ax in enumerate(axes.flat):
        if i < grids[name].shape[0]:
            ax.imshow(grids[name][i], cmap='viridis'); ax.set_title(f'L{i}', fontsize=8)
        ax.axis('off')
    fig.suptitle(f'{name} — channel-max of patch tokens along the residual stream')
    plt.tight_layout(); plt.show()

## 4. Streaming export over `stephen/` and `cat/`

Output layout per source folder (42 subfolders per model-side: 41 `layer_nn` + `input`):

```
p63_cache/stephen/input/frame_0000.png
p63_cache/stephen/plain/layer_00/frame_0000.png … layer_40/…
p63_cache/stephen/reg/layer_00/frame_0000.png   … layer_40/…
p63_cache/stephen/frames.json      # {"0": "0000.png", "1": "0001.png", ...}
p63_cache/cat/...
```

In [ ]:
def export_folder(folder_name):
    src = INPUT_ROOT / folder_name
    files = sorted(src.glob('*.png'))
    if not files:
        print(f'!! no pngs found in {src}'); return
    dst = OUT_ROOT / folder_name
    (dst / 'input').mkdir(parents=True, exist_ok=True)
    for name in MODELS:
        for i in range(N_LAYERS + 1):
            (dst / name / f'layer_{i:02d}').mkdir(parents=True, exist_ok=True)

    manifest = {}
    for m, f in enumerate(tqdm(files, desc=folder_name)):
        manifest[m] = f.name
        img_m, x_m = load_image(f)
        x_m = x_m.to(device)
        img_m.save(dst / 'input' / f'frame_{m:04d}.png')
        for name, model in MODELS.items():
            g = max_grids(get_residual_stream(model, x_m), N_SKIP[name])
            for i in range(g.shape[0]):
                save_viridis_png(g[i], dst / name / f'layer_{i:02d}' / f'frame_{m:04d}.png')
    json.dump(manifest, open(dst / 'frames.json', 'w'), indent=2)
    print(f'{folder_name}: {len(files)} frames × {len(MODELS)} models × {N_LAYERS + 1} layers  → {dst}')

for folder in STREAM_FOLDERS:
    export_folder(folder)

## 5. Sanity check what landed on disk

In [ ]:
for name in MODELS:
    s = np.load(stream_dir / f'stream_{name}.npy', mmap_mode='r')
    print(f'stream_{name}.npy  shape={s.shape} dtype={s.dtype}')
print('max_grids pngs:', {name: len(list((grid_dir / name).glob('*.png'))) for name in MODELS})

for folder in STREAM_FOLDERS:
    dst = OUT_ROOT / folder
    if not dst.exists():
        continue
    n_in = len(list((dst / 'input').glob('*.png')))
    n_layer_dirs = {name: len(list((dst / name).glob('layer_*'))) for name in MODELS}
    n_pngs = {name: len(list((dst / name).glob('layer_*/frame_*.png'))) for name in MODELS}
    print(f'{folder}: {n_in} input frames, layer dirs {n_layer_dirs}, frame pngs {n_pngs}')